# Giải mã tính mùa vụ

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC VÀ CHUẨN BỊ DỮ LIỆU
# ==========================================
df_sales = pd.read_csv('../dataset/sales.csv')
df_sales['Date'] = pd.to_datetime(df_sales['Date'])

# Tính Profit cho từng dòng dữ liệu
df_sales['Profit'] = df_sales['Revenue'] - df_sales['COGS']

# Trích xuất Năm và Tháng
df_sales['Year'] = df_sales['Date'].dt.year
df_sales['Month'] = df_sales['Date'].dt.month

# ==========================================
# 2. TẠO BẢNG TỔNG HỢP (PIVOT TABLES)
# ==========================================
# Tổng hợp theo Tháng và Năm cho cả Revenue và Profit
monthly_revenue = df_sales.groupby(['Year', 'Month'])['Revenue'].sum().unstack(level=0)
monthly_profit = df_sales.groupby(['Year', 'Month'])['Profit'].sum().unstack(level=0)

# ==========================================
# 3. TRỰC QUAN HÓA EDA
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(16, 20))

# --- BIỂU ĐỒ 1: HEATMAP LỢI NHUẬN (BẢN ĐỒ NHIỆT PROFIT) ---
# Biểu đồ này giúp xác định những "tháng vàng" mang lại tiền lời thực tế
ax1 = axes[0]
sns.heatmap(monthly_profit.T, annot=True, fmt=',.0f', cmap='RdYlGn', ax=ax1, cbar_kws={'label': 'Lợi Nhuận (VND)'})

ax1.set_title('EDA: BẢN ĐỒ NHIỆT LỢI NHUẬN THEO THÁNG & NĂM', fontsize=15, fontweight='bold')
ax1.set_xlabel('Tháng', fontsize=12)
ax1.set_ylabel('Năm', fontsize=12)

# --- BIỂU ĐỒ 2: SO SÁNH BIÊN ĐỘ REVENUE VS PROFIT (TRUNG BÌNH CÁC NĂM) ---
# Giúp tìm ra tính mùa vụ của hiệu suất sinh lời
ax2 = axes[1]
avg_stats = df_sales.groupby('Month')[['Revenue', 'Profit']].mean()

# Vẽ Revenue
ax2.plot(avg_stats.index, avg_stats['Revenue'], marker='o', color='#1f77b4', linewidth=3, label='Doanh Thu TB (Revenue)')
# Vẽ Profit
ax2.fill_between(avg_stats.index, avg_stats['Profit'], color='#2ca02c', alpha=0.3, label='Miền Lợi Nhuận TB (Profit)')
ax2.plot(avg_stats.index, avg_stats['Profit'], marker='s', color='#2ca02c', linewidth=3)

# Tính và hiển thị Biên lợi nhuận (Profit Margin %) trung bình từng tháng
for i, row in avg_stats.iterrows():
    margin = (row['Profit'] / row['Revenue']) * 100
    ax2.text(i, row['Profit'] - (ax2.get_ylim()[1]*0.05), f"{margin:.1f}%", 
             ha='center', fontsize=10, fontweight='bold', color='#155724')

ax2.set_title('EDA: TƯƠNG QUAN DOANH THU & LỢI NHUẬN TRUNG BÌNH THEO THÁNG', fontsize=15, fontweight='bold')
ax2.set_xticks(range(1, 13))
ax2.set_ylabel('Giá trị (VND)', fontsize=12)
ax2.set_xlabel('Tháng trong năm', fontsize=12)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

# ==========================================
# 4. TÌM KIẾM "KEYS" TỪ LỢI NHUẬN
# ==========================================
best_profit_month = avg_stats['Profit'].idxmax()
best_margin_month = (avg_stats['Profit'] / avg_stats['Revenue']).idxmax()

print(f"--- CHÌA KHÓA LỢI NHUẬN (PROFIT KEYS) ---")
print(f"Key 5 (Tháng kiếm lời tốt nhất): Tháng {best_profit_month}")
print(f"Key 6 (Tháng có biên lợi nhuận cao nhất): Tháng {best_margin_month}")
print(f"Key 7 (Hiệu quả mùa vụ): Nếu Biên lợi nhuận (%) ở biểu đồ 2 đi ngang -> Bạn kiểm soát giá vốn tốt.")
print(f"Key 8 (Rủi ro): Nếu doanh thu tăng nhưng % biên lợi nhuận giảm mạnh -> Bạn đang quá phụ thuộc vào giảm giá.")

# Phân tích sự vượt trội của tháng 5

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC VÀ CHUẨN BỊ DỮ LIỆU
# ==========================================
df_traffic = pd.read_csv('../dataset/web_traffic.csv')
df_traffic['date'] = pd.to_datetime(df_traffic['date'])
df_traffic['month'] = df_traffic['date'].dt.month

# Lọc riêng dữ liệu Tháng 5 và các tháng còn lại để đối chiếu
df_m5 = df_traffic[df_traffic['month'] == 5]
df_others = df_traffic[df_traffic['month'] != 5]

# ==========================================
# 2. TÍNH TOÁN TỶ TRỌNG NGUỒN TRAFFIC
# ==========================================
# Gom nhóm theo nguồn traffic cho Tháng 5
m5_source = df_m5.groupby('traffic_source')['sessions'].sum().reset_index()
# Gom nhóm theo nguồn traffic cho các tháng khác (để so sánh cấu trúc)
others_source = df_others.groupby('traffic_source')['sessions'].sum().reset_index()

# ==========================================
# 3. TRỰC QUAN HÓA
# ==========================================
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Biểu đồ 1: Cơ cấu nguồn Traffic Tháng 5 ---
axes[0].pie(m5_source['sessions'], labels=m5_source['traffic_source'], 
            autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'),
            explode=[0.05] * len(m5_source))
axes[0].set_title('Cấu Trúc Nguồn Traffic - THÁNG 5', fontsize=14, fontweight='bold')

# --- Biểu đồ 2: So sánh hiệu suất Traffic Source (Sessions vs Bounce Rate) ---
# Giúp biết nguồn nào "xịn" nhất (vào nhiều nhưng ít thoát)
m5_source_stats = df_m5.groupby('traffic_source').agg({
    'sessions': 'sum',
    'bounce_rate': 'mean'
}).reset_index()

ax2 = axes[1]
sns.scatterplot(data=m5_source_stats, x='sessions', y='bounce_rate', 
                hue='traffic_source', s=300, palette='deep', ax=ax2)

# Thêm nhãn tên nguồn vào các điểm
for i, txt in enumerate(m5_source_stats['traffic_source']):
    ax2.annotate(txt, (m5_source_stats['sessions'].iloc[i], m5_source_stats['bounce_rate'].iloc[i]),
                 xytext=(10, 10), textcoords='offset points', fontweight='bold')

ax2.set_title('Chất Lượng Từng Nguồn Traffic (Tháng 5)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Tổng lượt Sessions')
ax2.set_ylabel('Tỷ lệ thoát (Bounce Rate)')

plt.tight_layout()
plt.show()

# --- TỔNG KẾT KEY CHIẾN THUẬT ---
top_source = m5_source.loc[m5_source['sessions'].idxmax(), 'traffic_source']
best_quality_source = m5_source_stats.loc[m5_source_stats['bounce_rate'].idxmin(), 'traffic_source']

print(f"--- CHIẾN THUẬT QUẢNG CÁO THÁNG 5 ---")
print(f"Key 11 (Nguồn kéo khách chính): {top_source}")
print(f"Key 12 (Nguồn khách chất lượng nhất - ít thoát nhất): {best_quality_source}")